# Instruction tuning end to end: template → labels → LoRA → QLoRA

Chapters 8–9 · Day 12 · **DGX Spark GPU first**

This notebook gathers the instruction-tuning ideas from our SLP3 Chapter 8 discussions
into one flow. Each part builds on the last, with the real **Qwen3-0.6B** model:

| Part | Question | Learning note |
|---|---|---|
| A · Data | What exactly does the model see, and which tokens does it learn to write? | [topics 1–3](../../learning_artifacts/day-11-instruction-data-engineering/learning-instruction-tuning.md) |
| B · LoRA ideas | What are rank, approximation and low-rank approximation, and why might a fine-tuning update be low-rank? | topic 4 |
| C · Full vs. LoRA | What changes when only a low-rank update trains? | topic 4 |
| D · QLoRA | How can the frozen model be stored in 4 bits and still be trained through? | topic 5 |

Two ideas run through all of it. **Instruction tuning decides *what* the model learns from:**
instruction → response pairs with loss on the response. **PEFT decides *which* weights may move.**
They're independent choices, so Part C trains the *same* masked data both ways.

**Environment.** Use the `Python (DGX Spark Native)` kernel with CUDA. The Qwen3-0.6B tokenizer
and weights must be in the local Hugging Face cache; nothing is downloaded. QLoRA in Part D is
built by hand in PyTorch (`bitsandbytes` isn't installed on the Spark), so every step is
visible. An optional cell uses `bitsandbytes` if it's ever installed.

**Scale and limits.** The training runs are a few dozen steps on a small authored dataset. They
show *mechanics*: masks, gradients, memory, parameter counts. They don't show capability gains
on a real benchmark. Numbers can vary slightly between GPU runs; checks use tolerances.

In [ ]:
import gc, math, time, hashlib, itertools
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import transformers, peft
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "Use the Python (DGX Spark Native) kernel: this notebook is GPU first."
DEVICE = "cuda"
MODEL = "Qwen/Qwen3-0.6B"
torch.manual_seed(0)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.family": "DejaVu Sans",
                     "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
tok = AutoTokenizer.from_pretrained(MODEL, local_files_only=True)

def load_model(dtype=torch.bfloat16):
    return AutoModelForCausalLM.from_pretrained(MODEL, local_files_only=True, dtype=dtype).to(DEVICE)

def free():
    """Release GPU memory after `del`-ing models at the top level."""
    gc.collect(); torch.cuda.empty_cache()

def gib(n_bytes):
    return n_bytes / 2**30

print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__)

---

# Part A · From a conversation to graded tokens

Instruction tuning is ordinary next-token training with two changes: the data is
instruction → response pairs, and the loss counts only the response. Before any training we
need to see exactly what the model receives. Notebook Day 11 · 05 covers these steps in more
detail; here they set up the data for Parts C and D.

In [ ]:
conversation = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "Give me one word for a very happy feeling."},
    {"role": "assistant", "content": "Elated."},
    {"role": "user",      "content": "Another one?"},
    {"role": "assistant", "content": "Joyful."},
]

def render_ids(msgs, **kwargs):
    return list(tok.apply_chat_template(msgs, return_dict=True, **kwargs)["input_ids"])

HEADER = tok.encode("<|im_start|>assistant\n", add_special_tokens=False)
EMPTY_THINK = tok.encode("<think>\n\n</think>\n\n", add_special_tokens=False)
IM_END = tok.convert_tokens_to_ids("<|im_end|>")

def build_labels(ids):
    """Copy each assistant answer and its <|im_end|> into labels; -100 everywhere else.
    The empty think block is template scaffolding and stays ungraded (Day 11 · 05)."""
    labels, i = [-100] * len(ids), 0
    while i < len(ids):
        if ids[i:i + len(HEADER)] == HEADER:
            start = i + len(HEADER)
            if ids[start:start + len(EMPTY_THINK)] == EMPTY_THINK:
                start += len(EMPTY_THINK)
            end = ids.index(IM_END, start)
            labels[start:end + 1] = ids[start:end + 1]
            i = end + 1
        else:
            i += 1
    return labels

### Prediction

## Predict 1 · What the model sees and what it is graded on

Render the five-message conversation with Qwen3's own template and build the labels. How many tokens are graded, and which ones? Does the `\n` after each `<|im_end|>` get a label?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
ids = render_ids(conversation)
labels = build_labels(ids)
graded = [tok.decode([t]) for t, l in zip(ids, labels) if l != -100]
print(tok.decode(ids))
print("graded tokens:", graded)
assert graded == ["El", "ated", ".", "<|im_end|>", "Joy", "ful", ".", "<|im_end|>"]
assert labels[-1] == -100

rows, row = [], []
for t, l in zip(ids, labels):
    row.append((tok.decode([t]).replace("\n", "\\n"), l != -100))
    if t == IM_END:
        rows.append(row); row = []
rows[-1].extend(row)
fig, ax = plt.subplots(figsize=(12, 0.62 * len(rows) + 0.9))
for r, cells_ in enumerate(rows):
    x = 0.0
    for piece, on in cells_:
        w = 0.11 * len(piece) + 0.22
        ax.add_patch(plt.Rectangle((x, -r - 0.38), w, 0.76, edgecolor="#CBD5E1", linewidth=1,
                                   facecolor="#2563EB" if on else "#F1F5F9"))
        ax.text(x + w / 2, -r, piece, ha="center", va="center", fontsize=9, family="DejaVu Sans Mono",
                color="white" if on else "#111827")
        x += w + 0.05
ax.set(xlim=(-0.1, 15.5), ylim=(-len(rows) + 0.4, 0.6), xticks=[], yticks=[])
ax.set_title("Graded label tokens in blue; every other position is -100", loc="left")
for side in ax.spines.values():
    side.set_visible(False)
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

Eight tokens are graded: `Elated.` and `Joyful.` with their closing `<|im_end|>`. The
system prompt, user turns, role headers, Qwen3's empty think block and the trailing newline are
all `-100`. The model reads them as context but is never asked to produce them.

This is what makes the method **supervised**: each input comes with a designated target
(learning note topic 2). It isn't about who wrote the data. The grading of `<|im_end|>` is how the
model learns to stop (topic 3).

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Qwen3 chat template with graded label tokens](../figures/chapter-09/day-12-03_instruction_tuning_end_to_end-01.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

### Prediction

## Predict 2 · The shift, with the real model

Run Qwen3-0.6B on this conversation. Cross-entropy with `ignore_index=-100` should average over how many positions? Predict whether forgetting the one-position shift gives an error, or a number.

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
model = load_model()
x = torch.tensor([ids], device=DEVICE)
y = torch.tensor([labels], device=DEVICE)
with torch.no_grad():
    logits = model(x).logits.float()[0]

shifted = F.cross_entropy(logits[:-1], y[0, 1:], ignore_index=-100)     # logit t predicts token t+1
keep = y[0, 1:] != -100
manual = -torch.log_softmax(logits[:-1], -1)[keep].gather(1, y[0, 1:][keep][:, None]).mean()
hf_loss = model(x, labels=y).loss.float()                               # HF shifts labels inside
unshifted = F.cross_entropy(logits, y[0], ignore_index=-100)            # forgot the shift
print(f"graded positions: {int(keep.sum())} | shifted: {shifted:.4f} | manual: {manual:.4f} | "
      f"HF internal: {hf_loss:.4f} | unshifted (wrong task): {unshifted:.4f}")
assert int(keep.sum()) == 8
torch.testing.assert_close(shifted, manual, rtol=1e-4, atol=1e-4)
torch.testing.assert_close(shifted, hf_loss, rtol=2e-2, atol=2e-2)

### Reference explanation

### Reference answer and mechanism

Eight positions contribute. The hand-shifted loss equals the manual average over graded
positions, and Hugging Face's internal loss agrees: it shifts `labels` itself, so you pass
*unshifted* labels to the model. (Small differences come from bf16 arithmetic.)

Forgetting the shift doesn't raise an error. It returns a number with valid shapes, but each logit
is then graded on its *own* input token. That's a different task, and a silent one.

### Prediction

## Predict 3 · Masked does not mean untouched

Prompt tokens get no loss. Do the *embedding rows* of prompt-only tokens such as `Give` or `happy` receive a gradient? And do the *logits* at masked positions receive one?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
model.zero_grad(set_to_none=True)
emb = model.get_input_embeddings()
inputs = emb(x).detach().requires_grad_(True)              # watch gradients at each input position
out = model(inputs_embeds=inputs, labels=y)
out.logits.retain_grad()
out.loss.backward()

logit_grad = out.logits.grad[0].float().abs().sum(-1)      # per position
masked_positions = (y[0, 1:] == -100).nonzero().flatten()  # logit t predicts token t+1
print("max |dL/dlogit| at masked positions:", float(logit_grad[masked_positions].max()))
input_grad = inputs.grad[0].float().norm(dim=-1)
for word in ["Give", " happy", " feeling"]:
    pos = [i for i, t in enumerate(ids) if tok.decode([t]) == word][0]
    print(f"input-gradient norm at prompt token {word!r} (position {pos}): {float(input_grad[pos]):.3e}")
assert float(logit_grad[masked_positions].max()) == 0.0
assert float(input_grad[:20].max()) > 0
model.zero_grad(set_to_none=True)

### Reference explanation

### Reference answer and mechanism

The **logits** at masked positions get exactly zero gradient: those predictions aren't graded.
But the **inputs** at prompt positions get nonzero gradient. The graded answer `Elated.`
depends on the prompt through attention, so the loss flows back into how the prompt was
represented, and into every shared weight that processed it.

So the loss mask controls *what is graded*, not *what is trained*. Day 12 · 01 shows the same
effect in a tiny model.

**Training data for Parts C and D.** A small authored instruction set with checkable answers:
naming the color in a sentence and reversing two words. Every example is rendered with the
Qwen3 template and masked with `build_labels`. Held-out examples use combinations not seen in
training.

In [ ]:
colors = ["red", "blue", "green", "yellow", "purple", "orange", "black", "white"]
things = ["ball", "box", "kite", "cup", "hat", "car"]
words = ["sun", "moon", "tree", "river", "stone", "cloud", "bird", "lamp"]
pairs = []
for i, (c, t) in enumerate(itertools.product(colors, things)):
    pairs.append((f"Reply with only the color: The {t} is {c}.", c, i % 6 == 0))
for i, (a, b) in enumerate(itertools.permutations(words[:6], 2)):
    pairs.append((f"Reverse the two words: {a} {b}", f"{b} {a}", i % 6 == 0))
train = [(q, a) for q, a, held in pairs if not held]
heldout = [(q, a) for q, a, held in pairs if held]

def encode(q, a):
    ids_ = render_ids([{"role": "user", "content": q}, {"role": "assistant", "content": a}])
    return ids_, build_labels(ids_)

def batch(examples):
    enc = [encode(q, a) for q, a in examples]
    n = max(len(i) for i, _ in enc)
    pad = tok.pad_token_id
    ids_ = torch.full((len(enc), n), pad); lab = torch.full((len(enc), n), -100); att = torch.zeros((len(enc), n), dtype=torch.long)
    for k, (i, l) in enumerate(enc):
        ids_[k, :len(i)] = torch.tensor(i); lab[k, :len(l)] = torch.tensor(l); att[k, :len(i)] = 1
    return {"input_ids": ids_.to(DEVICE), "labels": lab.to(DEVICE), "attention_mask": att.to(DEVICE)}

TRAIN, HELD = batch(train), batch(heldout)
print(len(train), "training and", len(heldout), "held-out examples; graded tokens per batch:",
      int((TRAIN["labels"] != -100).sum()), "/", int((HELD["labels"] != -100).sum()))
print("example:", train[0])

---

# Part B · The ideas behind LoRA

SLP3 §8.2: "with LoRA we freeze these layers and instead update a *low-rank approximation*".
Three words to make concrete: **rank**, **approximation**, **low-rank approximation**
(learning note topic 4). Then one question: why should a fine-tuning update be low-rank at all?

### Prediction

## Predict 4 · Rank: how many independent patterns?

The matrix `[[1,2,3],[2,4,6],[3,6,9]]` has 9 numbers. What is its rank? Add a second, independent pattern to it: what happens to the rank?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
col = torch.tensor([[1.], [2.], [3.]])
row = torch.tensor([[1., 2., 3.]])
M1 = col @ row                                      # column x row
print(M1)
print("rank:", int(torch.linalg.matrix_rank(M1)), "| numbers stored as column + row:", col.numel() + row.numel())

M2 = M1 + torch.tensor([[1.], [0.], [-1.]]) @ torch.tensor([[0., 1., 0.]])
print(M2)
print("rank after adding a second pattern:", int(torch.linalg.matrix_rank(M2)))
assert int(torch.linalg.matrix_rank(M1)) == 1 and int(torch.linalg.matrix_rank(M2)) == 2

### Reference explanation

### Reference answer and mechanism

Rank 1: every row is a multiple of `[1 2 3]`, so 3 + 3 = 6 numbers rebuild all 9. Adding a
second, independent column × row pattern raises the rank to 2. **A rank-r matrix is a sum of r
column × row patterns.** That's the whole idea LoRA builds on.

### Prediction

## Predict 5 · Approximation: keep the strongest patterns

Take a real photograph as a matrix and keep only its top r patterns (singular values). Roughly how many patterns out of hundreds do you need before the picture is recognizable?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
img = plt.imread(plt.matplotlib.cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))
gray = torch.tensor(img[..., :3].mean(-1), dtype=torch.float32, device=DEVICE) / 255.0
U, S, Vh = torch.linalg.svd(gray, full_matrices=False)
ranks = [1, 5, 20, 50]
fig, axes = plt.subplots(1, len(ranks) + 2, figsize=(15, 3.2), gridspec_kw={"width_ratios": [1] * (len(ranks) + 1) + [1.4]})
errors = {}
for ax, r in zip(axes, ranks + [len(S)]):
    approx = (U[:, :r] * S[:r]) @ Vh[:r]
    errors[r] = float((approx - gray).norm() / gray.norm())
    ax.imshow(approx.clamp(0, 1).cpu(), cmap="gray"); ax.set_axis_off()
    ax.set_title(f"r = {r}" + (" (full)" if r == len(S) else f"\nerror {errors[r]:.1%}"), fontsize=9)
rs = torch.arange(1, len(S) + 1)
err_curve = torch.sqrt(1 - torch.cumsum(S**2, 0) / (S**2).sum()).cpu()
axes[-1].plot(rs, err_curve, color="#2563EB"); axes[-1].set(xlabel="patterns kept (r)", ylabel="relative error", xscale="log")
axes[-1].set_title("error vs. patterns kept", fontsize=9)
plt.tight_layout(); plt.show()
print("image shape:", tuple(gray.shape), "| relative errors:", {r: round(e, 3) for r, e in errors.items()})

### Reference explanation

### Reference answer and mechanism

A few dozen patterns out of hundreds already give a recognizable face, and the error curve drops
steeply at first, then flattens. This is an **approximation**: simpler, close but not exact. Most
real matrices behave this way, with a few strong patterns and many weak ones. Truncating the
singular value decomposition keeps the strong ones; for a given r it's the best possible rank-r
approximation in this error measure.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Low-rank approximations of a photograph](../figures/chapter-09/day-12-03_instruction_tuning_end_to_end-02.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

### Prediction

## Predict 6 · Low-rank approximation: parameters vs. error

A Qwen3 projection is 1024 × 1024 (`v_proj`). Writing it as tall-thin × short-wide with rank r costs `2·1024·r` numbers. What fraction is that at r = 8? Will a *pretrained* weight be as compressible as the photo?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
W = model.model.layers[10].self_attn.v_proj.weight.detach().float()
d_out, d_in = W.shape
Sw = torch.linalg.svdvals(W)
Sr = torch.linalg.svdvals(torch.randn_like(W) * W.std())
rank_grid = [1, 4, 8, 16, 64, 256, 512]
w_err = [float(torch.sqrt(1 - (Sw[:r]**2).sum() / (Sw**2).sum())) for r in rank_grid]
r_err = [float(torch.sqrt(1 - (Sr[:r]**2).sum() / (Sr**2).sum())) for r in rank_grid]
params = [r * (d_out + d_in) for r in rank_grid]
print(f"full matrix: {d_out * d_in:,} numbers")
for r, p, e in zip(rank_grid, params, w_err):
    print(f"r = {r:>4}: {p:>9,} numbers ({p / (d_out * d_in):6.1%}) | rank-r error of the pretrained weight: {e:.1%}")
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(params, w_err, "o-", color="#2563EB", label="pretrained v_proj (layer 10)")
ax.plot(params, r_err, "s--", color="#64748B", label="random matrix, same shape")
ax.axvline(d_out * d_in, color="#B45309", lw=1); ax.text(d_out * d_in, 0.9, " full size", color="#B45309")
ax.set(xscale="log", xlabel="numbers stored (r · (d_out + d_in))", ylabel="relative error", ylim=(0, 1.02))
ax.legend(frameon=False); plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

At r = 8 a rank-r factorization stores 16,384 numbers, **1.6%** of the 1,048,576 in the full
matrix. But the pretrained weight is *not* very compressible: its rank-8 error is large, much
closer to a random matrix than to the photo. Pretrained weights carry many meaningful patterns.

That's why LoRA doesn't compress `W`. It keeps `W` frozen and complete, and makes only the
**update** low-rank: `W + (α/r)·BA` in the course's notation (SLP3 writes `W + AB`). The bet is
that the *change* needed for a new task has few strong patterns, even though `W` doesn't.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Rank-r error vs parameters for a pretrained weight and a random matrix](../figures/chapter-09/day-12-03_instruction_tuning_end_to_end-03.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

### Prediction

## Predict 7 · Why a fine-tuning update can be low-rank

For a linear layer `y = W h`, the gradient is `∂L/∂W = Σ_t δ_t h_tᵀ`, a sum of one column × row pattern per token. For one training example of n tokens, what is the largest possible rank of this gradient?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
q, a = train[0]
one = batch([(q, a)])
n_tokens = int(one["attention_mask"].sum())
model.float()                                       # fp32 for this check: bf16 rounding adds tiny spurious directions
model.zero_grad(set_to_none=True)
model(**one).loss.backward()
G = model.model.layers[10].self_attn.v_proj.weight.grad.float()
print(f"tokens in the example: {n_tokens} | gradient shape: {tuple(G.shape)} | "
      f"numerical rank: {int(torch.linalg.matrix_rank(G))}")
assert int(torch.linalg.matrix_rank(G)) <= n_tokens
model.zero_grad(set_to_none=True)
model.to(torch.bfloat16)                            # exact round trip: the weights started as bf16

### Reference explanation

### Reference answer and mechanism

At most n: one column × row pattern per token. A 1024 × 1024 gradient from one short example has
rank no larger than its token count. Each SGD step therefore adds a low-rank change, and a small
dataset touches only a limited set of directions.

This makes LoRA's bet plausible, not guaranteed. Adam rescales each entry, and many steps over
many examples add up. Part C measures the actual full fine-tuning update to check.

---

# Part C · The same SFT data, two ways: full fine-tuning vs. LoRA

The data and masks stay fixed (Part A). Only the set of weights allowed to move changes.
**Full fine-tuning** updates every weight. **LoRA** freezes them all and trains small `A` and
`B` matrices on `q_proj` and `v_proj` (rank 8, α = 16), the course's Chapter 9 setup.

Parts C and D use one transparent LoRA layer, so that LoRA and QLoRA differ *only* in how the
frozen base weight is stored. `peft` serves as a cross-check of the parameter count.

In [ ]:
def nf4_levels():
    """NF4 from its definition: normal quantiles, 8 positive, 7 negative and an exact zero."""
    offset = 0.5 * (1 - 1 / (2 * 15) + 1 - 1 / (2 * 16))
    pos = torch.special.ndtri(torch.linspace(offset, 0.5, 9, dtype=torch.float64)[:-1])
    neg = -torch.special.ndtri(torch.linspace(offset, 0.5, 8, dtype=torch.float64)[:-1])
    v = torch.cat([pos, torch.zeros(1, dtype=torch.float64), neg]).sort().values
    return (v / v.abs().max()).float()

NF4 = nf4_levels()

def quantize_blocks(w, levels, block=64):
    """Block-wise absmax quantization: each block of `block` weights gets its own scale."""
    flat = w.float().reshape(-1, block)
    absmax = flat.abs().amax(1, keepdim=True).clamp_min(1e-12)
    codes = (flat / absmax).unsqueeze(-1).sub(levels.to(w.device)).abs().argmin(-1).to(torch.uint8)
    return codes, absmax.squeeze(1)

def dequantize_blocks(codes, absmax, levels, shape):
    return (levels.to(codes.device)[codes.long()] * absmax[:, None]).reshape(shape)

class FrozenQuantizedMatmul(torch.autograd.Function):
    """y = h Wᵀ for a frozen 4-bit W. Nothing dequantized is kept for backward: the weight is
    rebuilt from its codes when the input gradient is needed, as QLoRA's kernels do."""
    @staticmethod
    def forward(ctx, h, layer):
        ctx.layer = layer
        return F.linear(h, layer.base_weight())
    @staticmethod
    def backward(ctx, grad_out):
        return grad_out @ ctx.layer.base_weight(), None

class LoRALinear(nn.Module):
    """Frozen base weight (dense bf16, or packed NF4) plus an optional trainable (α/r)·B·A."""
    def __init__(self, base, r=8, alpha=16, quantize=False, block=64):
        super().__init__()
        self.shape, self.quantized, self.scale = tuple(base.weight.shape), quantize, (alpha / r if r else 0.0)
        w = base.weight.detach()
        if quantize:
            codes, absmax = quantize_blocks(w, NF4, block)
            self.register_buffer("packed", ((codes[:, 0::2] << 4) | codes[:, 1::2]).contiguous())  # 2 codes/byte
            self.register_buffer("absmax", absmax.float())
            self.register_buffer("levels", NF4.to(w.device))
        else:
            self.register_buffer("weight", w.clone())
        self.A = nn.Parameter(torch.randn(r, self.shape[1], device=w.device) / math.sqrt(self.shape[1])) if r else None
        self.B = nn.Parameter(torch.zeros(self.shape[0], r, device=w.device)) if r else None   # B = 0 at start
    def base_weight(self):
        if not self.quantized:
            return self.weight
        codes = torch.stack([self.packed >> 4, self.packed & 15], -1).reshape(self.packed.shape[0], -1)
        return dequantize_blocks(codes, self.absmax, self.levels, self.shape).to(torch.bfloat16)  # just in time
    def forward(self, h):
        y = FrozenQuantizedMatmul.apply(h, self) if self.quantized else F.linear(h, self.weight)
        if self.A is not None:
            y = y + (F.linear(F.linear(h.float(), self.A), self.B) * self.scale).to(y.dtype)
        return y

def add_lora(model, quantize=False, targets=("q_proj", "v_proj"), r=8):
    for layer in model.model.layers:
        for parent in (layer.self_attn, layer.mlp):
            for name, child in list(parent.named_children()):
                if isinstance(child, nn.Linear):
                    setattr(parent, name, LoRALinear(child, r=r if name in targets else 0, quantize=quantize))
    for p in model.parameters():
        p.requires_grad_(False)
    for m in model.modules():
        if isinstance(m, LoRALinear) and m.A is not None:
            m.A.requires_grad_(True); m.B.requires_grad_(True)
    gc.collect(); torch.cuda.empty_cache()
    return model

def heldout_loss(m):
    with torch.no_grad():
        return float(m(**HELD).loss)

def train_run(m, lr, steps=40):
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    before, losses, t0 = heldout_loss(m), [], time.time()
    for _ in range(steps):
        loss = m(**TRAIN).loss
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        losses.append(float(loss))
    torch.cuda.synchronize()
    stored = sum(t.numel() * t.element_size() for t in [*m.parameters(), *m.buffers()])
    return dict(train_losses=losses, heldout_before=before, heldout_after=heldout_loss(m),
                seconds=time.time() - t0, peak_gib=gib(torch.cuda.max_memory_allocated()),
                model_gib=gib(stored), trainable=sum(p.numel() for p in params))

### Prediction

## Predict 8 · How many weights train, and does LoRA change the model at step 0?

Count trainable parameters for full fine-tuning and for rank-8 LoRA on `q_proj`/`v_proj` across 28 layers. Then: before any training, does adding LoRA change the model's output?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
total = sum(p.numel() for p in model.parameters())
with torch.no_grad():
    plain_logits = model(**HELD).logits.float()
lora_model = add_lora(model)                       # wraps the same loaded model in place
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
grad_tensors = [n for n, p in lora_model.named_parameters() if p.requires_grad]
print(f"tensors that receive gradients: {len(grad_tensors)} (28 layers x q/v x A/B), e.g. {grad_tensors[:2]}")
assert len(grad_tensors) == 28 * 2 * 2 and all(n.endswith((".A", ".B")) for n in grad_tensors)
with torch.no_grad():
    lora_logits = lora_model(**HELD).logits.float()
per_layer = 8 * (1024 + 2048) + 8 * (1024 + 1024)  # q: A 8x1024 + B 2048x8 ; v: A 8x1024 + B 1024x8
print(f"full fine-tuning trains {total:,} parameters")
print(f"LoRA trains {trainable:,} = 28 x {per_layer:,} ({trainable / total:.3%} of the model)")
print("max |logit change| from adding LoRA before training:", float((lora_logits - plain_logits).abs().max()))

cfg = peft.LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.0)
peft_model = peft.get_peft_model(load_model(), cfg)
peft_count = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"peft LoraConfig(r=8, q_proj, v_proj) trainable parameters: {peft_count:,}")
assert trainable == 28 * per_layer == peft_count
assert float((lora_logits - plain_logits).abs().max()) == 0.0
del lora_model, model, peft_model; free()

### Reference explanation

### Reference answer and mechanism

Full fine-tuning trains all ~596 million parameters (the embedding is shared with the output
head). LoRA trains **1,146,880**, about **0.19%**, and `peft` counts exactly the same. Only 112
tensors receive gradients: an `A` and a `B` for `q_proj` and `v_proj` in each of 28 layers.
Every pretrained weight, including the embedding, stays frozen.

Before training, LoRA changes nothing. `B` starts at zero, so `(α/r)·B·A = 0` and the wrapped
model gives bit-identical logits. Training starts exactly from the pretrained model and moves
away from it only through the low-rank update.

### Prediction

## Predict 9 · Train both on the same masked data

Run 40 steps of full fine-tuning (fp32 weights, AdamW, lr 1e-5) and of LoRA (bf16 frozen base, lr 1e-3) on the same batch. Which uses more GPU memory, and by roughly how much? Will LoRA's held-out loss fall as far as full fine-tuning's?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
runs = {}
full = load_model(torch.float32)
W_before = {n: p.detach().clone() for n, p in full.named_parameters()
            if n.endswith(("layers.10.self_attn.v_proj.weight", "layers.10.self_attn.q_proj.weight"))}
torch.manual_seed(0); runs["full fine-tuning"] = train_run(full, lr=1e-5)
delta_full = {n: (dict(full.named_parameters())[n].detach() - W_before[n]).float() for n in W_before}
del full; free()

torch.manual_seed(0); lora_model = add_lora(load_model())
runs["LoRA (bf16 base)"] = train_run(lora_model, lr=1e-3)
lora_v = lora_model.model.layers[10].self_attn.v_proj
delta_lora = (lora_v.B @ lora_v.A * lora_v.scale).detach().float()
del lora_model, lora_v; free()

for name, r in runs.items():
    print(f"{name:<18} trainable {r['trainable']:>11,} | peak GPU memory {r['peak_gib']:5.2f} GiB | "
          f"model storage {r['model_gib']:5.2f} GiB | held-out loss {r['heldout_before']:.3f} -> {r['heldout_after']:.2e} | {r['seconds']:.1f} s")
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for name, color in (("full fine-tuning", "#111827"), ("LoRA (bf16 base)", "#2563EB")):
    ax.plot(runs[name]["train_losses"], color=color, label=name)
ax.set(xlabel="optimizer step", ylabel="training loss (graded tokens)", yscale="log"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()
assert runs["LoRA (bf16 base)"]["peak_gib"] < runs["full fine-tuning"]["peak_gib"]

### Reference explanation

### Reference answer and mechanism

Both runs fit the small task, and both held-out losses fall close to zero. In the reference run,
full fine-tuning peaked at about **19 GiB** and LoRA at about **10.6 GiB**. Full fine-tuning keeps
fp32 weights, a gradient and two AdamW moments for all ~596M parameters. LoRA keeps those extras
for only 1.1M.

Why isn't LoRA's peak far lower still? Most of the remaining memory is **activations**, not
weights. The output layer scores all 151,936 vocabulary entries at every position of every
example, and the backward pass needs those activations. LoRA shrinks the trainable state, but not
the activations.

The loss curves compare *recipes* (different parameter spaces, learning rates and dtypes),
not a controlled capability claim. On a 40-step toy task both can fit. Chapter 9's Full400 vs.
LoRA400 comparison is the course's real-model treatment.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Training loss for full fine-tuning and LoRA on the same masked data](../figures/chapter-09/day-12-03_instruction_tuning_end_to_end-04.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

### Prediction

## Predict 10 · Was the full fine-tuning update low-rank?

Take the full fine-tuning change `ΔW` of layer 10's `v_proj` and compare its singular values with LoRA's update and with a random matrix. What fraction of ΔW's energy do its top 8 directions hold?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
def energy_top(M, r):
    s = torch.linalg.svdvals(M)
    return float((s[:r]**2).sum() / (s**2).sum())

dW = delta_full[[n for n in delta_full if "v_proj" in n][0]]
rand = torch.randn_like(dW)
print(f"top-8 energy | full FT dW: {energy_top(dW, 8):.1%} | LoRA dW: {energy_top(delta_lora, 8):.1%} | "
      f"random: {energy_top(rand, 8):.1%}")
print("numerical rank of LoRA dW:", int(torch.linalg.matrix_rank(delta_lora)))
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for M, label, color in ((dW, "full fine-tuning ΔW", "#111827"), (delta_lora, "LoRA ΔW = (α/r)BA", "#2563EB"),
                        (rand, "random matrix", "#64748B")):
    s = torch.linalg.svdvals(M).cpu()
    ax.plot(torch.arange(1, 65), (s / s[0])[:64], label=label, color=color)
ax.axvline(8, color="#B45309", lw=1); ax.text(8.5, 0.92, "r = 8", color="#B45309")
ax.set(xlabel="singular value index", ylabel="relative singular value", ylim=(0, 1.05)); ax.legend(frameon=False)
plt.tight_layout(); plt.show()
assert int(torch.linalg.matrix_rank(delta_lora)) <= 8

### Reference explanation

### Reference answer and mechanism

LoRA's update has rank at most 8 by construction. In the reference run, the full fine-tuning
update put about **70%** of its energy in its top 8 directions, against about **3%** for a random
matrix of the same shape, which spreads energy evenly.

So in this run the full update was itself dominated by a few directions, and a rank-8 update
could capture most of it. That's the bet LoRA makes, measured rather than assumed. The remaining
~30% is what a rank-8 update can't express, so rank is still a real constraint. This is one layer,
one tiny dataset and 40 steps: evidence about this run, not a general law.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Singular value spectra of full fine-tuning and LoRA updates](../figures/chapter-09/day-12-03_instruction_tuning_end_to_end-05.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

---

# Part D · QLoRA: store the frozen base in 4 bits

LoRA made *training* cheap, but the frozen base still sits in memory at 16 bits. QLoRA stores
it in 4 bits and keeps only the small adapters at higher precision (learning note topic 5).
Every piece below is built by hand: block-wise scales, NF4 levels, packing, just-in-time
dequantization.

### Prediction

## Predict 11 · Why quantize in blocks?

Quantize a real Qwen weight matrix to 16 levels with one global scale, and with one scale per block of 64 weights. Then plant a single large outlier weight. Which scheme does the outlier hurt, and why?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
model = load_model()
W = model.model.layers[10].self_attn.q_proj.weight.detach().float()
def rel_err(Wq, W):
    return float((Wq - W).norm() / W.norm())
uniform = torch.linspace(-1, 1, 16)
results = {}
W_outlier = W.clone(); W_outlier[0, 0] = 50.0          # one planted outlier weight
for label, M in (("clean", W), ("with one outlier", W_outlier)):
    for scheme, block in (("global scale", M.numel()), ("blocks of 64", 64)):
        c, s = quantize_blocks(M, uniform, block)
        results[(label, scheme)] = rel_err(dequantize_blocks(c, s, uniform, M.shape), M)
for k, v in results.items():
    print(f"{k[0]:<17} {k[1]:<13} relative error {v:.1%}")
assert results[("with one outlier", "global scale")] > results[("with one outlier", "blocks of 64")]

### Reference explanation

### Reference answer and mechanism

With one global scale, the largest weight sets the step size for *every* weight. Even on the
clean matrix the error is several times worse than with blocks (about 47% vs. 10% in the
reference run). Most weights are small and crowd onto the few levels near zero.

A single planted outlier makes the global version collapse completely. Sixteen evenly spaced
levels from −1 to 1 have no exact zero, so every small weight is pushed to ±1/15 of the outlier's
size. With a scale per block of 64, the outlier only coarsens its own block. Every other block
keeps fine resolution, at the cost of one stored scale per 64 weights.

### Prediction

## Predict 12 · Why NF4 levels?

Trained weights are roughly bell-shaped. Build NF4's 16 levels from normal quantiles and compare with 16 evenly spaced levels on the same block-normalized weights. Which has lower error, and where does NF4 put its levels?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
PUBLISHED_NF4 = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534,
    0.16093020141124725, 0.24611230194568634, 0.33791524171829224, 0.44070982933044434,
    0.5626170039176941, 0.7229568362236023, 1.0])                # bitsandbytes' NF4 table
print("max |our NF4 - published NF4|:", float((NF4 - PUBLISHED_NF4).abs().max()))
errs = {}
for name, lv in (("uniform levels", uniform), ("NF4 levels", NF4)):
    c, s = quantize_blocks(W, lv)
    errs[name] = rel_err(dequantize_blocks(c, s, lv, W.shape), W)
print({k: f"{v:.2%}" for k, v in errs.items()})
normed = (W.reshape(-1, 64) / W.reshape(-1, 64).abs().amax(1, keepdim=True)).flatten().cpu()
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(normed.numpy(), bins=200, color="#CBD5E1", density=True, label="block-normalized weights")
for lv, color, y, name in ((uniform, "#64748B", 0.15, "uniform"), (NF4, "#2563EB", 0.35, "NF4")):
    ax.plot(lv.numpy(), np.full(16, y), "|", ms=18, mew=2, color=color, label=f"{name} levels")
ax.set(xlabel="weight / block absmax", ylabel="density"); ax.legend(frameon=False, loc="upper right")
plt.tight_layout(); plt.show()
assert float((NF4 - PUBLISHED_NF4).abs().max()) < 1e-6 and errs["NF4 levels"] < errs["uniform levels"]

### Reference explanation

### Reference answer and mechanism

NF4 built from its definition matches the published table to about 10⁻⁷, and gives lower error
than evenly spaced levels on this real weight matrix. The histogram shows why: most normalized
weights sit near zero, and NF4 places its levels densely there and sparsely in the tails. Even
levels waste resolution on rare large values.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![NF4 and uniform 4-bit levels over block-normalized Qwen weights](../figures/chapter-09/day-12-03_instruction_tuning_end_to_end-06.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

### Prediction

## Predict 13 · Memory accounting

Estimate bytes for full fine-tuning, LoRA and QLoRA for 0.6B and 7B parameters (weights, gradients and AdamW state only). Then check the real storage of our quantized layers, including the per-block scales and double quantization.

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
def estimate(n_params, lora_params, n_quantizable):
    full = n_params * (4 + 4 + 8)                      # fp32 weights, grads, two AdamW moments
    lora = n_params * 2 + lora_params * (4 + 4 + 8)    # bf16 frozen base + fp32 adapter state
    qlora = (n_params - n_quantizable) * 2 + n_quantizable * 0.5 + n_quantizable / 64 * 1 \
            + n_quantizable / 64 / 256 * 4 + lora_params * 16   # NF4 + 8-bit scales + fp32 2nd-level scales
    return full, lora, qlora
linear_params = sum(m.weight.numel() for layer in model.model.layers for m in layer.modules() if isinstance(m, nn.Linear))
total = sum(p.numel() for p in model.parameters())
rows = {"Qwen3-0.6B": estimate(total, 1_146_880, linear_params),
        "7B (illustrative, 90% quantizable)": estimate(7e9, 20e6, 0.9 * 7e9)}
for name, (f_, l_, q_) in rows.items():
    print(f"{name:<36} full {gib(f_):6.1f} GiB | LoRA {gib(l_):5.2f} GiB | QLoRA {gib(q_):5.2f} GiB")

codes, absmax = quantize_blocks(W, NF4)
bits_plain = 4 + 32 / 64
blocks = absmax.reshape(-1, 256)
mean = absmax.mean()
scale2 = (blocks - mean).abs().amax(1, keepdim=True) / 127
q8 = torch.round((blocks - mean) / scale2).clamp(-127, 127)
absmax_dq = (q8 * scale2 + mean).flatten()
bits_double = 4 + 8 / 64 + 32 / (64 * 256)
err_dq = rel_err(dequantize_blocks(codes, absmax_dq, NF4, W.shape), W) - rel_err(dequantize_blocks(codes, absmax, NF4, W.shape), W)
print(f"bits per weight: NF4 + fp32 scales {bits_plain:.3f} | with double-quantized scales {bits_double:.3f} "
      f"(saves {bits_plain - bits_double:.3f}) | extra relative error from 8-bit scales: {err_dq:+.4%}")
print(f"quantizable linear weights in Qwen3-0.6B: {linear_params:,} of {total:,} ({linear_params / total:.0%}); "
      "the tied embedding stays bf16")
del model; free()

### Reference explanation

### Reference answer and mechanism

Full fine-tuning needs about 16 bytes per parameter, LoRA about 2, and QLoRA about 0.5 on the
quantized part. For a 7B model that's roughly 100 GiB vs. 13 vs. 4. These are weights and
optimizer state only; activations come on top.

Double quantization stores each block's scale in 8 bits instead of 32, cutting the overhead
from 0.5 to about 0.127 bits per weight (≈ 0.37 saved), at a negligible error cost here. One
detail matters for small models: Qwen3-0.6B's tied embedding (~155M parameters) isn't a linear
layer we quantize, so QLoRA's saving is smaller than for large models.

### Prediction

## Predict 14 · QLoRA on Qwen3-0.6B, by hand

Quantize all 196 linear layers to packed NF4 (blocks of 64), attach the same rank-8 LoRA on `q_proj`/`v_proj`, and train 40 steps on the same masked data with the same learning rate as bf16 LoRA. Predict: how close is the loss curve to bf16 LoRA? What happens to the 4-bit codes during training?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
torch.manual_seed(0)
qmodel = add_lora(load_model(), quantize=True)
storage = {"packed 4-bit codes": sum(b.numel() for n, b in qmodel.named_buffers() if n.endswith("packed")),
           "fp32 block scales": sum(b.numel() * 4 for n, b in qmodel.named_buffers() if n.endswith("absmax"))}
bf16_linear_bytes = 2 * sum(m.shape[0] * m.shape[1] for m in qmodel.modules() if isinstance(m, LoRALinear))
codes_before = hashlib.sha256(b"".join(b.cpu().numpy().tobytes() for n, b in qmodel.named_buffers() if n.endswith("packed"))).hexdigest()
runs["QLoRA (NF4 base)"] = train_run(qmodel, lr=1e-3)
codes_after = hashlib.sha256(b"".join(b.cpu().numpy().tobytes() for n, b in qmodel.named_buffers() if n.endswith("packed"))).hexdigest()
print({k: f"{gib(v) * 1024:.0f} MiB" for k, v in storage.items()}, f"| same layers in bf16: {gib(bf16_linear_bytes) * 1024:.0f} MiB")
print("4-bit codes unchanged by training:", codes_before == codes_after)
for name, r in runs.items():
    print(f"{name:<18} trainable {r['trainable']:>11,} | peak GPU memory {r['peak_gib']:5.2f} GiB | "
          f"model storage {r['model_gib']:5.2f} GiB | held-out loss {r['heldout_before']:.3f} -> {r['heldout_after']:.2e} | {r['seconds']:.1f} s")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for name, color in (("full fine-tuning", "#111827"), ("LoRA (bf16 base)", "#2563EB"), ("QLoRA (NF4 base)", "#047857")):
    axes[0].plot(runs[name]["train_losses"], color=color, label=name)
axes[0].set(xlabel="optimizer step", ylabel="training loss", yscale="log"); axes[0].legend(frameon=False)
names = list(runs)
axes[1].bar(names, [runs[n]["peak_gib"] for n in names], color=["#111827", "#2563EB", "#047857"])
axes[1].set(ylabel="peak GPU memory during training (GiB)")
plt.tight_layout(); plt.show()
assert codes_before == codes_after
assert runs["QLoRA (NF4 base)"]["model_gib"] < runs["LoRA (bf16 base)"]["model_gib"]
assert runs["QLoRA (NF4 base)"]["peak_gib"] < runs["LoRA (bf16 base)"]["peak_gib"]

### Reference explanation

### Reference answer and mechanism

The 196 linear layers shrink from about 840 MiB in bf16 to about 210 MiB of 4-bit codes plus
about 26 MiB of scales. The whole model's storage halves, from about 1.11 to 0.52 GiB; the bf16
embedding is the part that doesn't shrink. Training never touches the codes: their hash is
identical after 40 steps. Only `A` and `B` learn, exactly as in bf16 LoRA.

The quantized model starts from a *higher* held-out loss than the bf16 model (about 3.5 vs. 2.6
in the reference run): the forward pass runs on a slightly blurred copy of the weights. The
adapters then learn the task just as well. Quantization costs some starting accuracy, and the
fine-tuning partly works around it.

**A lesson from building it.** The first version of this layer simply called
`F.linear(h, dequantized_W)`. Its peak memory was *no lower* than bf16 LoRA, because autograd
kept every dequantized bf16 weight alive for the backward pass, undoing the 4-bit saving.
`FrozenQuantizedMatmul` keeps nothing: it rebuilds the weight from its codes when the input
gradient is needed. That trades extra compute (slower steps) for memory, the same trade QLoRA's
real kernels make. Peak memory then falls below bf16 LoRA (about 10.0 vs. 10.6 GiB here). The gap
is modest because activations dominate this small model's peak; QLoRA's advantage grows with
model size.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Training loss and peak GPU memory for full fine-tuning, LoRA and QLoRA](../figures/chapter-09/day-12-03_instruction_tuning_end_to_end-07.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

**Optional: the library version.** If `bitsandbytes` is installed in this kernel, the next cell
loads Qwen3-0.6B in its NF4 format and counts memory. It's skipped otherwise, and nothing above
depends on it.

In [ ]:
import importlib.util
if importlib.util.find_spec("bitsandbytes") is None:
    print("bitsandbytes is not installed in this kernel: optional library cross-check skipped.")
else:
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.bfloat16)
    m4 = AutoModelForCausalLM.from_pretrained(MODEL, local_files_only=True, quantization_config=bnb, device_map={"": 0})
    print(f"bitsandbytes NF4 model footprint: {gib(m4.get_memory_footprint()):.2f} GiB")
    del m4; free()

## Reading the evidence

**What this notebook establishes, for this run:** Qwen3's template and the reference labels grade
only answers and `<|im_end|>`. The loss mask zeroes gradients at masked logits but not through
the prompt. A rank-r matrix is a sum of r patterns, and pretrained weights are not very
compressible. One example's gradient has rank at most its token count. Rank-8 LoRA on Q/V trains
0.19% of the weights, starts from the exact pretrained model, and uses far less memory than full
fine-tuning. Block-wise NF4, built from its definition, matches the published table and beats
evenly spaced levels on real weights. QLoRA keeps the 4-bit codes fixed and reduces memory
further.

**What it doesn't establish:** that LoRA or QLoRA matches full fine-tuning in quality, or any
capability gain. The tasks are tiny and authored, the runs are 40 steps, and the recipes differ in
learning rate and dtype. The low-rank measurement covers one layer of one run. Memory estimates
cover weights and optimizer state only. GPU results vary slightly between runs.

**Where to go next:** Chapter 9 §9.8 for LoRA's mathematics and the real Full400/LoRA400
comparison; the [learning note](../../learning_artifacts/day-11-instruction-data-engineering/learning-instruction-tuning.md)
for the discussions behind each part.